In [ ]:
import re 
import polars as pl 

In [ ]:
transcript9 = open("/Users/sabrinapereira/Desktop/Text Analysis/TextAnalysis_2026/Session9_Transcript.txt").read() '''bla'''

transcript9

In [ ]:
chunks = re.split(r"\n(?=\w[\w\s]*:)", transcript9)

chunks

In [ ]:
rows = []
for chunk in chunks:
    match = re.match(r"^(\w[\w\s]*):\s+(.+?)(?:\s+(\d{2}:\d{2}:\d{2}[\.\d]*))?$", chunk, re.DOTALL) # find the groups
    if match:
        rows.append(match.groups())  

df = pl.DataFrame(rows, schema=["speaker", "speech", "time"])

print(df.head(10))
print(df.dtypes) 
print(df.describe)

In [ ]:
print(df["speaker"].unique())


In [ ]:
df = df.with_columns([
    pl.col("speaker")
        .str.replace("MArk", "Mark")
        .alias("speaker")
])

print(df["speaker"].unique())

In [ ]:
def get_event(speech):
    if re.search(r'\*1\*', speech):
        return "Reading Story"
    if re.search(r'\*2\*', speech):
        return "Reading Poem"
    if re.search(r'<.+?>', speech, re.DOTALL):  # re.DOTALL so < > can span lines
        return "Citing Text"
    if re.search(r'".+?"', speech, re.DOTALL):  # same for quotes spanning lines
        return "Citing GM"
    return None

In [ ]:
df = df.with_columns(
    pl.col("speech").map_elements(get_event, return_dtype=pl.String).alias("event")
)

df.describe()

In [ ]:
with pl.Config(tbl_rows=50, fmt_str_lengths=200, tbl_width_chars=1000):
    print(df.filter(pl.col("event").is_not_null()))

In [ ]:

def get_event(speech):
    if re.search(r'\*1\*', speech):
        complete = len(re.findall(r'\*1\*', speech)) % 2 == 0
        return "Reading Story", complete
    if re.search(r'\*2\*', speech):
        complete = len(re.findall(r'\*2\*', speech)) % 2 == 0
        return "Reading Poem", complete
    if re.search(r'[<>]', speech):
        complete = bool(re.search(r'<', speech) and re.search(r'>', speech))
        return "Citing Text", complete
    if re.search(r'"', speech):
        complete = len(re.findall(r'"', speech)) % 2 == 0
        return "Citing GM", complete
    return None, False

In [ ]:
events = []
current_event = None

for speaker, speech, time in df[["speaker", "speech", "time"]].iter_rows():
    event, is_complete = get_event(speech)

    if event is not None:
        if current_event == event:
            current_event = None
        elif not is_complete:
            current_event = event
    else:
        event = current_event

    events.append(event)

df = df.with_columns(pl.Series("event", events))

df.describe() 

In [ ]:
with pl.Config(tbl_rows=50, fmt_str_lengths=200, tbl_width_chars=1000):
    print(df.filter(pl.col("event").is_not_null()))

In [ ]:
rows = []
current_event = None 

for chunk in chunks:
    match = re.match(r"^(\w[\w\s]*):\s+(.+?)(?:\s+(\d{2}:\d{2}:\d{2}[\.\d]*))?$", chunk, re.DOTALL)
    if match:
        speaker, speech, time = match.groups()
        event, is_complete = get_event(speech)

        if event is not None:
            if current_event == event:

                current_event = None
            elif not is_complete:

                current_event = event

        else:

            event = current_event

        rows.append((speaker, speech, time, event))

df = pl.DataFrame(
    rows,
    schema=["speaker", "speech", "time", "event"],
    orient="row"
)

df.describe() 

In [ ]:
with pl.Config(tbl_rows=50, fmt_str_lengths=200, tbl_width_chars=1000):
    print(df.filter(pl.col("event").is_not_null()))

In [ ]:
full_text = " ".join(df["speech"].drop_nulls().to_list()) 
issues = []

counts = {
    "*1*": len(re.findall(r'\*1\*', full_text)),
    "*2*": len(re.findall(r'\*2\*', full_text)),
    '"':   len(re.findall(r'"', full_text)),
    "<":   len(re.findall(r'<', full_text)),
    ">":   len(re.findall(r'>', full_text)),
}


if counts["*1*"] % 2 != 0: 
    issues.append(f"odd number of *1* markers ({counts['*1*']})")
if counts["*2*"] % 2 != 0:
    issues.append(f"odd number of *2* markers ({counts['*2*']})")
quote = '"'
if counts['"'] % 2 != 0:
    issues.append(f"odd number of {quote} markers ({counts[quote]})")
if counts["<"] != counts[">"]:
    issues.append(f"{counts['<']} opening < but {counts['>']} closing >")

    if counts["<"] > counts[">"]: 
        closed = True
        issue = False
        for row in df.iter_rows(named=True):
            if not row["speech"]:
                continue
            for char in row["speech"]:
                if (char == "<") and (closed == True):
                    closed = False
                elif (char == ">") and (closed == False):
                    closed = True
                elif (char == "<") and (closed == False):
                    issues.append(f"unclosed before this: '{row['speech']}'")
                                
    if counts[">"] > counts["<"]: 
        opened = False
        for row in df.iter_rows(named=True):
            if not row["speech"]:
                continue
            for char in row["speech"]:
                if (char == "<") and (opened == False):
                    opened =True
                elif (char == ">") and (opened == True):
                    opened = False
                elif (char == "<") and (opened == True):
                    issues.append(f"unopened before this: '{row['speech']}'")




if not issues:
    print("All markers are balanced!")
else:
    print(f"Found {len(issues)} issue(s):\n")
    for issue in issues:
        print(issue)
